# 非评分实验：多类别分类——一对多

当类别数超过两个时，一对多是一种选择方法。  
![pic](./figures/onevsmany.png)

## 目标
在本实验中，你将：
- 使用已经开发的函数（compute_cost、compute_gradient、predict、gradient_descent）执行二项（是/否）分类
- 编写一个多分类预测例程，在 n 个二项决策之间进行选择
- 使用决策边界绘图逻辑

# 大纲
- [工具](#tools)
- [数据集](#dataset)
- [一对多实现](#ova)


# 多类别分类：一对多（OVA）
在本实验中，我们将探索当类别数超过两个时，如何使用一对多方法进行分类。这项技术是我们一直在使用的二类别或二项逻辑回归的扩展。

在二项逻辑回归中，我们训练一个模型，判断样本属于某个类别还是不属于该类别。一对多（OVA）通过训练 $n$ 个模型扩展了此方法。每个模型负责识别一个类别。训练给定类别的模型时，会重新构造训练集，将该类别标记为正类，所有其他类别标记为负类。进行预测时，每个样本会由全部 $n$ 个模型处理，并选择预测输出最大的模型。

在本实验中，我们将构建一个 OVA 分类器。
## 工具
- 我们将利用此前的工作来构建和训练模型。这些例程已经提供。
- 绘制决策边界和数据集很有帮助。生成这些图相当复杂，因此下面提供了辅助例程。
        - plot_mc_decision_boundary() 将使用你在本作业中编写的预测例程 `predict_mc`
- 我们将创建一个多类别数据集，并使用常用的 [`SkLearn`](https://scikit-learn.org/stable/) 例程。

In [ ]:
from lab_utils import *
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
import copy
import math

这些例程已经提供，但研究它们的工作方式很有启发性。绘图例程经常会用到许多较为少见但很实用的 NumPy 例程。绘制决策边界会使用 `matplotlib's` 等高线图。等高线图会在数值发生变化的边界处绘制一条线，此功能可用于描绘决策的变化。简而言之，该例程分为 3 个步骤：
- 在二维网格中创建位置密集的细网格，并构建这些点的数组。
- 对每个点进行预测。在本例中，这包括对最佳预测进行投票。
- 使用等高线图绘制网格与预测结果（`Z`）。

In [ ]:
#Plot a multi-class decision boundary
def plot_mc_decision_boundary(X,nclasses, W, b , predict_mc_function, class_labels=None, legend=False):

    # create a mesh to points to plot
    h = 0.1  # step size in the mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    points = np.c_[xx.ravel(), yy.ravel()]

    #make predictions for each point in mesh
    Z = predict_mc_function(points,W,b)
    Z = Z.reshape(xx.shape)

    #contour plot highlights boundaries between values - classes in this case
    plt.figure()
    plt.contour(xx, yy, Z, colors='g') 
    plt.axis('tight')


In [ ]:
# Plot  multi-class training points
def plot_mc_data(X, y, class_labels=None, legend=False):
    classes = np.unique(y)
    for i in classes:
        label = class_labels[i] if class_labels else "class {}".format(i)
        idx = np.where(y == i)
        plt.scatter(X[idx, 0], X[idx, 1],  cmap=plt.cm.Paired,
                    edgecolor='black', s=20, label=label)
    if legend: plt.legend()

我们提供了你在此前实验中开发的例程，用于创建并拟合/训练模型。你可以随意将它们替换为自己的版本。（请保留一份原始版本，以防万一。）

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, predict_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn theta. Updates theta by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      X :    (array_like Shape (m,n)
      y :    (array_like Shape (m,1))
      w_in : (array_like Shape (n,1)) Initial values of parameters of the model
      b_in : (scalar)                 Initial value of parameter of the model
      cost_function:                  function to compute cost
      gradient_function:              function to compute the gradient
      predict_function                function to compute the output of the model
      alpha : (float)                 Learning rate
      num_iters : (int)               number of iterations to run gradient descent
    Returns
      w : (array_like Shape (n,)) Updated values of parameters of the model after
          running gradient descent
      b : (scalar)                Updated value of parameter of the model after
          running gradient descent
    """
    
    # number of training examples
    m = len(X)
    
    # An array to store cost J and w's at each iteration primarily for graphing later
    J_history = []
    w_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dJdb,dJdw = gradient_function(X, y, w, b, predict_function)   ##None

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dJdw               ##None
        b = b - alpha * dJdb               ##None

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            cost =  cost_function(X, y, w, b)
            J_history.append(cost)

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters/10) == 0:
            w_history.append(w)
            print(f"Iteration {i:4}: Cost {float(J_history[-1]):8.2f}   ")
    #print(w,b,len(J_history), len(w_history) )   
    return w, b, J_history, w_history #return w and J,w history for graphing

In [ ]:
def compute_cost_logistic_matrix(X, y, w, b):
    """
    Computes the cost over all examples
    Args:
      X : (array_like Shape (m,n)) data, m examples by n features
      y : (array_like Shape (m,1)) target value 
      w : (array_like Shape (n,1)) Values of parameters of the model      
      b : (array_like Shape (n,1)) Values of bias parameter of the model
    Returns:
      total_cost: (scalar)         cost 
    """
    m = X.shape[0]
    
    f = sigmoid(X @ w + b)
    total_cost = (1/m)*(np.dot(-y.T, np.log(f)) - np.dot((1-y).T, np.log(1-f)))
    
    return total_cost[0,0]

In [ ]:
def predict_logistic_matrix(X, w, b): 
    """
    single predict using linear regression
    Args:
      x : (array_like Shape (m,n)) feature values house size, bedrooms..
      w : (array_like Shape (n,)) parameters for prediction   
      b : (scalar               ) parameter  for prediction   
    Returns
      p: ((array_like Shape (n,)  predictions
    """
    p = sigmoid(X @ w + b )         
    
    return(p)    

In [ ]:
def predict_thresh(X, w, b, threshold=0.5): 
    """
    Predict whether the label is 0 or 1 using learned logistic
    regression parameters w, b. Threshold the output of the sigmoid to predict 1 or 0.
    
    Parameters
    ----------
    X : array_like
        Shape (m, n) 
    
    w : array_like
        Parameters of the model
        Shape (n, 1)
    b : scalar
    
    Returns
    -------

    p: array_like
        Shape (m,)
        The predictions for X using a threshold at 0.5
    """
    # number of training examples
    m = X.shape[0]   
    p = np.zeros(m)
   
    for i in range(m):
        f_w = sigmoid(np.dot(w.T, X[i])+b)
        p[i] = f_w >= threshold
    
    return p

In [ ]:
def compute_gradient_logistic_matrix(X, y, w, b, predict_function): 
    """
    Computes the gradient for linear regression 
 
    Args:
      X : (array_like Shape (m,n)) variable such as house size 
      y : (array_like Shape (m,1)) actual value 
      w : (array_like Shape (n,1)) Values of parameters of the model      
      b : (scalar )                Values of parameter of the model      
      predict_function: (function) function to call to make prediction
    Returns
      dJdw: (array_like Shape (n,1)) The gradient of the cost w.r.t. the parameters w. 
      dJdb: (scalar)                 The gradient of the cost w.r.t. the parameter b. 
                                  
    """
    m,n = X.shape
    f_wb = predict_function(X, w, b) 
    err  = f_wb - y                 
    dJdw = (1/m) * (X.T @ err)     
    dJdb = (1/m) * np.sum(err)     
        
    return dJdb,dJdw

<a name='dataset'></a>
## 数据集
下面，我们使用 `SkLearn` 工具创建 3 个数据“簇”。这是一种快速、简便的分类数据生成方法。借助 NumPy 的 [`np.unique`](https://numpy.org/doc/stable/reference/generated/numpy.unique.html)，可以查看类别的数量和取值。
- **注意**：我们将创建 3 个类别。

In [ ]:
# make 3-class dataset for classification
centers = [[-5, 0], [0, 4.5], [5, -1]]
X_train, y_train = make_blobs(n_samples=500, centers=centers, cluster_std=0.85,random_state=40)


In [ ]:
plot_mc_data(X_train,y_train,["blob one", "blob two", "blob three"], legend=True)
plt.show()

In [ ]:
# show classes in data set
print(f"unique classes {np.unique(y_train)}")
# show how classes are represented
print(f"unique classes {y_train[:10]}")
# show shapes of our dataset
print(f"shape of X_train: {X_train.shape}, shape of y_train: {y_train.shape}")

创建一对多训练集时，需要针对每个类别从 `y_train` 创建一个“二元”训练集。对于该类别中的所有样本，这个集合中的值设为 `1`。NumPy 提供了一个便于完成此操作的功能，如下所示：

In [ ]:
y_cat_2 = (y_train == 2) + 0
print(y_cat_2[:10])

你将需要二进制值 (0,1)。

In [ ]:
y_cat_2 = (y_train == 2).astype(float)
print(y_cat_2[:10])

<a name='ova'></a>
## 一对多实现

你将分三个步骤实现 OVA 算法。
- 创建并训练三个“模型”，每个模型都经过训练，用于选择三个类别中的一个。
- 创建一个例程 `predict_mc`，它将使用这些模型进行预测并选择最佳答案。
- 使用预测例程绘制决策边界。

### 第 1 步：创建并训练 3 个模型。
其中涉及的步骤与过去使用梯度下降的实验类似。
对于每个类别：
- 分离与当前类别关联的目标 `y`。
- 创建参数初始值 `w_init` 和 `b_init`。
- 调用梯度下降。alpha=1e-2、num_iters=1000 的效果很好。
    - gradient_descent 调用包含多个参数。该例程位于上方。请使用*提示*中的代码仔细检查你的解答。请注意，梯度下降如何利用你目前为止开发的所有例程。
    - 你将调用 `predict_logistic_matrix` 来执行预测，其中不包含阈值逻辑。训练时不使用阈值；随后使用模型时再添加阈值。
    - 这将返回参数 $w$ 和 $b$。$w$ 和 $b$ 构成你的*模型*，你会将它们存入数组中。重要的是，该数组中*每个模型对应一列*。这种排列方式允许使用矩阵运算，你将在后续实验中逐渐熟悉它。
- 使用训练数据和模型（$w$、$b$）调用 predict，以绘制训练结果。

下面有一个遍历各类别的 for 循环，它会：
- 创建目标数组，将当前类别设为 1，其他所有类别设为 0。
- 执行你的代码
- 绘制这种数据解释
- 绘制预测值
请用你的代码替换 `None`。

<details>
  <summary><font size="2" color="darkgreen"><b>提示</b></font></summary>

```python
classes=np.unique(y_train)   # three classes, [0,1,2]
m,n = X_train.shape          # number of examples, number of features
c = len(classes)             # number of classe

# storage for our models (w), one column per class
W_models = np.zeros((n,len(classes)))   
b_models = np.zeros(c)
plt.figure(figsize=(14, 14))             

for i in classes:
    yc = (y_train == classes[i]).astype(float)
    yc = yc.reshape(-1,1)  

    ### START CODE HERE ### 
    w_init = np.zeros((2,1))                                                               
    b_init = 0.                                                                            
    w_final, b_final,_,_ = gradient_descent(X_train, yc, w_init, b_init,                   
                                      compute_cost_logistic_matrix,                       
                                      compute_gradient_logistic_matrix,                  
                                      predict_logistic_matrix,                            
                                      alpha = 1e-2, num_iters=1000)                         
    W_models[:,i] = w_final[:,0]                                                          
    b_models[i] = b_final                                                                 
    pred =  predict_thresh(X_train, w_final,b_final )                                    

    ### END CODE HERE ###         

    #Left Side, training data in All vs i
    ax = plt.subplot(3,2, 2*i + 1)
    plot_mc_data(X_train, yc,legend=True); plt.title(f"Training Classes, class {i}"); 

    #Right Side, model i's prediction after training
    ax = plt.subplot(3,2, 2*i + 2)
    plot_mc_data(X_train,pred,legend=True); plt.title("Predicted Classes after training");
plt.show
```
</details>

In [ ]:
classes=np.unique(y_train)   # three classes, [0,1,2]
m,n = X_train.shape          # number of examples, number of features
c = len(classes)             # number of classe

# storage for our models (w), one column per class
W_models = np.zeros((n,len(classes)))   
b_models = np.zeros(c)
plt.figure(figsize=(14, 14))             

for i in classes:
    yc = (y_train == classes[i]).astype(float)
    yc = yc.reshape(-1,1)  

    ### START CODE HERE ### 

    w_init = None  
    b_init = None  
    # call gradient descent, double check your solution with Hint
    w_final, b_final,_,_ = None  
    ### END CODE HERE ###         
    W_models[:,i] = w_final[:,0]                                                           
    b_models[i] = b_final                                                                  
    pred =  predict_thresh(X_train, w_final,b_final )                                      

    #Left Side, training data in All vs i
    ax = plt.subplot(3,2, 2*i + 1)
    plot_mc_data(X_train, yc,legend=True); plt.title(f"Training Classes, class {i}"); 

    #Right Side, model i's prediction after training
    ax = plt.subplot(3,2, 2*i + 2)
    plot_mc_data(X_train,pred,legend=True); plt.title("Predicted Classes after training");
plt.show()

<details>
<summary>
    <b>**预期输出**：</b>
</summary>

 ![asdf](./figures/C1W3_trainvpredict.PNG)

现在，我们已经训练了 3 个模型，接下来将编写一个例程来选择最佳预测。回顾一下，该操作包括：
- 为每个模型进行预测
- 选出最大的预测值

-第 1 步：给定 $X$ 以及矩阵 `W_model` 和 `b_models`，执行预测，得到三个预测结果。如下图所示，可以采用向量化形式实现。这并不是一个简单的操作，值得花时间理解其作用。也可以使用 for 循环实现。
![图片](./figures/C1W3_mcpredict.PNG)  
-第 2 步：使用 `np.argmax(axis=1)` 返回预测值最高的**类别**。请注意，类别是 [0,1,2] 中的一个值，而 `np.argmax` 返回的索引恰好也是 [0,1,2] 中的一个值。

<details>
  <summary><font size="2" color="darkgreen"><b>提示</b></font></summary>

```python
def predict_mc(X,W,b, verbose = False):
    """
    Computes n predictions and selects the best.
    Args:
      X : (array_like Shape (m,n)) feature values used in prediction.  
      W : (array_like Shape (n,c)) Matrix of parameter. Each column represents 1  model
      b : (array_like Shape (c, )) vector of bias parameter. Each column represents 1  model
    Returns
      sclass: (array_like Shape (m,1)) The selected class the values belong in. Values 0 to c.
    """
    ### START CODE HERE ### 
    ### BEGIN SOLUTION ###  
    z_wb = X @ W + b               #Matrix multiply and add  ##None
    f_wb = sigmoid(z_wb)              #sigmoid                  ##None
    pclass = f_wb.argmax(axis=1)      #argmax                   ##None
    ### END SOLUTION ###  
    ### END CODE HERE ### 
    if verbose: print("z_wb.shape",z_wb.shape); print(z_wb)
    if verbose: print("pclass",pclass)
    return(pclass)
```
</details>

In [ ]:
def predict_mc(X,W,b, verbose = False):
    """
    Computes n predictions and selects the best.
    Args:
      X : (array_like Shape (m,n)) feature values used in prediction.  
      W : (array_like Shape (n,c)) Matrix of parameter. Each column represents 1  model
      b : (array_like Shape (c, )) vector of bias parameter. Each column represents 1  model
    Returns
      sclass: (array_like Shape (m,1)) The selected class the values belong in. Values 0 to c.
    """
    ### START CODE HERE ### 

    #Matrix multiply and add
    #sigmoid  
    #argmax
    ### END CODE HERE ### 
    if verbose: print("z_wb.shape",z_wb.shape); print(z_wb)
    if verbose: print("pclass",pclass)
    return(pclass)

In [ ]:
#Test your model
tmp_X = np.array([[-2.,-6.],[6,0],[-2,6]])                            #(2,2)
tmp_w = np.array([[-1.117, 0.103, 0.963], [-0.863, 1.155, -0.954]])   #(2,3)
tmp_b = np.array([-0.267 -1.4577 -0.238])                             #(3, )
print(tmp_X.shape, tmp_w.shape, tmp_b.shape)

tmp_fw = predict_mc(tmp_X,tmp_w,tmp_b, verbose = True)
print(tmp_fw)

<details>
<summary>
    <b>**预期输出**：</b>
</summary>

```
(3, 2) (2, 3) (1,)
z_wb.shape (3, 3)
[[ 5.4493 -9.0987  1.8353]
 [-8.6647 -1.3447  3.8153]
 [-4.9067  4.7613 -9.6127]]
pclass [0 2 1]
[0 2 1]
```

现在，我们可以对任意点进行预测，因此可以绘制一幅包含决策边界的图。`plot_mc_decision_boundary` 会调用你的 `predict_mc` 函数。

In [ ]:
#plot the decison boundary. Pass in our models - the w's and b's assocated with each model and predict_mc
plot_mc_decision_boundary(X_train,3, W_models, b_models, predict_mc)
plt.title("model decision boundary vs original training data")

#add the original data to the decison boundary
plot_mc_data(X_train,y_train,["blob one", "blob two", "blob three"], legend=True)
plt.show()

<details>
<summary>
    <b>**预期输出**：</b>
</summary>

![sdf](./figures/C1W3_boundary.PNG)

完成了！您现在已经构建了一个多分类器。

让我们再试一种情况，只需稍微移动这些数据簇：
## 第二个测试用例

In [ ]:
# make 3-class dataset for classification
centers = [[-5, 0], [0, 1], [5, -1]]
X_train, y_train = make_blobs(n_samples=500, centers=centers, cluster_std=1.2,random_state=40)


In [ ]:
plot_mc_data(X_train,y_train,["blob one", "blob two", "blob three"], legend=True)
plt.show()

In [ ]:
# show classes in data set
print(f"unique classes {np.unique(y_train)}")
# show shapes of our dataset
print(f"shape of X_train: {X_train.shape}, shape of y_train: {y_train.shape}")

观察上图，你是否发现当前方法存在任何潜在问题？

请整合上面的各个部分，或创建子例程，以生成类似第一个示例的决策边界图。

<details>
  <summary><font size="2" color="darkgreen"><b>提示</b></font></summary>

```python
classes=np.unique(y_train)   # three classes, [0,1,2]
m,n = X_train.shape          # number of examples, number of features
c = len(classes)             # number of classe

# storage for our models (w), one column per class
W_models = np.zeros((n,len(classes)))   
b_models = np.zeros(c)

for i in classes:
    yc = (y_train==classes[i]) + 0
    yc = yc.reshape(-1,1)

    w_init = np.zeros((2,1))   
    b_init = 0.
    w_final, b_final,_,_ = gradient_descent(X_train, yc, w_init, b_init,
                                      compute_cost_logistic_matrix, 
                                      compute_gradient_logistic_matrix, 
                                      predict_logistic_matrix,
                                      alpha = 1e-2, num_iters=1000)     
    W_models[:,i] = w_final[:,0]
    b_models[i] = b_final
    pred =  predict_thresh(X_train, w_final,b_final ) 

plot_mc_decision_boundary(X_train,3, W_models, b_models, predict_mc)
plt.title("model decision boundary vs original training data")

#add the original data to the decison boundary
plot_mc_data(X_train,y_train,["blob one", "blob two", "blob three"], legend=True)
plt.show()
```
</details>

In [ ]:
#Rewrite code here



<details>
<summary>
    <b>**预期输出**：</b>
</summary>

![asdf](./figures/C1W3_example2.PNG)
    
我们将在下一个实验中学习带多项式特征的逻辑回归。这将使我们能够处理纯线性解不足以解决的问题。

本 Notebook 参考了 scikit-learn.org 上的一个示例。作者是 Tom Dupre la Tour <tom.dupre-la-tour@m4x.org>